# Flow Matching 训练 CelebA（Colab 版）

这个 Notebook 把原始 `flowmatching.py` 拆成了适合 Colab 运行的分块版本，并补上了必要的中文注释。

注意：
1. `KaggleDatasetAdapter.HUGGING_FACE` 更适合读取 `csv / parquet / json / sqlite / excel` 这类表格文件。
2. `jessicali9530/celeba-dataset` 的图片实际存放在 `img_align_celeba.zip` 中，所以训练时需要先下载并解压图片。
3. 下面会同时演示：
   - 用 `kagglehub.dataset_download(...)` 下载整个 CelebA 数据集
   - 用 `KaggleDatasetAdapter.HUGGING_FACE` 读取 `list_eval_partition.csv` 这个分区表


## 1. 安装依赖

Colab 一般已经自带 `torch` 和 `torchvision`，这里只补安装 KaggleHub、HF 数据集支持，以及一些常用工具库。

In [ ]:
# 如果你在 Colab 里第一次运行，先安装依赖
!pip -q install "kagglehub[hf-datasets]" pandas pillow tqdm matplotlib


## 2. 可选：Kaggle 认证

如果公开数据集访问时要求认证，可以用 Colab Secrets 中的 `KAGGLE_API_TOKEN`，或者手动执行 `kagglehub.login()`。

In [ ]:
# 这一格是可选的。
# 1. 如果你在 Colab Secrets 中保存了 KAGGLE_API_TOKEN，这里会自动读取。
# 2. 如果还不行，再取消注释 kagglehub.login() 手动登录。

import os

try:
    from google.colab import userdata  # 仅在 Colab 中可用
    if "KAGGLE_API_TOKEN" not in os.environ:
        token = userdata.get("KAGGLE_API_TOKEN")
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
            print("已从 Colab Secrets 读取 KAGGLE_API_TOKEN")
except Exception:
    pass

# 如需手动登录，取消下面两行注释：
# import kagglehub
# kagglehub.login()


## 3. 导入库与全局配置

这里统一导入依赖，并设置随机种子、设备、输出目录等基础配置。

In [ ]:
import math
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils
from tqdm.auto import tqdm

import kagglehub
from kagglehub import KaggleDatasetAdapter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 强制检查 GPU 可用性
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

ROOT_DIR = Path("/content")
DATA_DIR = ROOT_DIR / "celeba_data"
EXTRACT_DIR = ROOT_DIR / "celeba_extracted"
SAVE_DIR = ROOT_DIR / "flowmatch_outputs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("save dir:", SAVE_DIR)

## 4. 超参数配置

为了让 Colab 更容易先跑通，这里默认采用一个比原始脚本更轻量的配置。流程验证通过以后，再逐步把参数调大。

In [ ]:
DATASET_HANDLE = "jessicali9530/celeba-dataset"
PARTITION_FILE = "list_eval_partition.csv"
IMAGE_ZIP_FILE = "img_align_celeba.zip"

batch_size = 16
lr = 1e-4
num_epochs = 5
image_size = 64
base_ch = 64
flow_steps = 100

# 先用训练集前 20000 张图片跑通流程；想全量训练时改成 None
max_train_images = 20_000

num_workers = min(2, os.cpu_count() or 0)
pin_memory = DEVICE.type == "cuda"
sample_every = 1
num_sample_images = 9
use_amp = DEVICE.type == "cuda"

print({
    "batch_size": batch_size,
    "lr": lr,
    "num_epochs": num_epochs,
    "image_size": image_size,
    "base_ch": base_ch,
    "flow_steps": flow_steps,
    "max_train_images": max_train_images,
    "num_workers": num_workers,
    "use_amp": use_amp,
})


## 5. 下载 CelebA，并用 Hugging Face 方式读取分区表

这里做两件事：
- 用 `dataset_download(...)` 下载整个数据集（因为图片在 zip 里）
- 用 `KaggleDatasetAdapter.HUGGING_FACE` 读取 `list_eval_partition.csv`（因为它本质上是表格元数据）


In [ ]:
dataset_dir = Path(
    kagglehub.dataset_download(
        DATASET_HANDLE,
        output_dir=str(DATA_DIR),
    )
)
print("dataset_dir:", dataset_dir)

# 递归列出所有文件，确认 zip 包实际位置
print("\n所有文件（递归）：")
for path in sorted(dataset_dir.rglob("*")):
    print(" -", path.relative_to(dataset_dir))

# 尝试定位 zip 文件
zip_candidates = list(dataset_dir.rglob("*.zip"))
print("\n找到的 zip 文件:", zip_candidates)

# 这里直接沿用你给的 load_dataset(...) 写法。
# 注意：在较新的 kagglehub 版本中，它内部会提示未来更推荐使用 dataset_load(...)。
partition_hf = kagglehub.load_dataset(
    KaggleDatasetAdapter.HUGGING_FACE,
    DATASET_HANDLE,
    PARTITION_FILE,
)

print(partition_hf)
partition_df = partition_hf.to_pandas()
partition_df.head()

## 6. 解压图片压缩包

CelebA 的图片在 `img_align_celeba.zip` 中。第一次运行时会解压，后续再次运行则会直接复用。

In [ ]:
# 数据集已经解压好了，直接定位图片目录
# 从 cell-10 的输出看，图片在 dataset_dir / "img_align_celeba" / "img_align_celeba" 下

image_root = dataset_dir / "img_align_celeba" / "img_align_celeba"

# 如果上面的路径不存在，尝试其他可能的位置
if not image_root.exists():
    image_root = dataset_dir / "img_align_celeba"
if not image_root.exists():
    first_jpg = next(dataset_dir.rglob("*.jpg"), None)
    if first_jpg is None:
        raise FileNotFoundError("没有找到任何 jpg 图片")
    image_root = first_jpg.parent

print("image_root:", image_root)
sample_names = [p.name for _, p in zip(range(5), image_root.glob("*.jpg"))]
print("sample images:", sample_names)

## 7. 构建训练集文件列表

这里利用分区表选择训练集。Kaggle 的 `list_eval_partition.csv` 中：`0=训练集`，`1=验证集`，`2=测试集`。

In [ ]:
partition_col = "partition" if "partition" in partition_df.columns else partition_df.columns[-1]
image_id_col = "image_id" if "image_id" in partition_df.columns else partition_df.columns[0]

train_names = partition_df.loc[partition_df[partition_col] == 0, image_id_col].tolist()
if max_train_images is not None:
    train_names = train_names[:max_train_images]

train_paths = [image_root / name for name in train_names]

if len(train_paths) == 0:
    raise RuntimeError("训练图片列表为空，请检查分区表或解压目录")

print("训练图片数量:", len(train_paths))
print("第一张图片:", train_paths[0])


## 8. 数据集类、DataLoader 与样例可视化

和原始脚本一样，图片会先经过 `Resize + CenterCrop + ToTensor()`，然后缩放到 `[-1, 1]`。

In [ ]:
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),  # 先转到 [0, 1]
])

class CelebADataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        # Flow Matching 训练时统一使用 [-1, 1] 范围
        img = img * 2.0 - 1.0
        return img

def to_display_range(x):
    return (x.clamp(-1, 1) + 1.0) / 2.0

def show_tensor_images(x, title="Sample Images"):
    x = to_display_range(x.detach().cpu())
    grid = utils.make_grid(
        x[:num_sample_images],
        nrow=int(math.sqrt(num_sample_images)),
        padding=2,
    )
    plt.figure(figsize=(6, 6))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.title(title)
    plt.show()

train_dataset = CelebADataset(train_paths, transform=transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=num_workers > 0,
)

sample_batch = next(iter(train_loader))
print("batch shape:", sample_batch.shape)
show_tensor_images(sample_batch, title="CelebA Training Batch")


## 9. 模型基础模块

下面保留原始脚本中的主要结构：时间嵌入、残差块、自注意力、下采样块、上采样块和中间块。

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # t 的形状是 (B,)，表示连续时间步
        device = t.device
        half = self.dim // 2
        emb_scale = -(math.log(10000) / max(half - 1, 1))
        emb = torch.exp(torch.arange(half, device=device) * emb_scale)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        )
        self.time_emb_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch),
        )
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        )
        self.residual_conv = (
            nn.Conv2d(in_ch, out_ch, kernel_size=1)
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x, t_emb):
        residual = self.residual_conv(x)
        h = self.conv1(x)
        t_emb = self.time_emb_proj(t_emb)
        h = h + t_emb[:, :, None, None]
        h = self.conv2(h)
        return h + residual

class SelfAttention2D(nn.Module):
    def __init__(self, in_channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(8, in_channels)
        self.qkv = nn.Conv2d(in_channels, in_channels * 3, kernel_size=1)
        self.proj_out = nn.Conv2d(in_channels, in_channels, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)

        q = q.view(B, self.num_heads, C // self.num_heads, H * W)
        k = k.view(B, self.num_heads, C // self.num_heads, H * W)
        v = v.view(B, self.num_heads, C // self.num_heads, H * W)

        attn = torch.softmax(
            torch.matmul(q.transpose(-2, -1), k) / math.sqrt(C // self.num_heads),
            dim=-1,
        )
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        out = out.contiguous().view(B, C, H, W)
        out = self.proj_out(out)
        return x + out

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_blocks=2, downsample=True, use_attention=False):
        super().__init__()
        self.blocks = nn.ModuleList([
            ResidualBlock(in_ch if i == 0 else out_ch, out_ch, time_emb_dim)
            for i in range(num_blocks)
        ])
        self.attn = SelfAttention2D(out_ch) if use_attention else nn.Identity()
        self.downsample = (
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=2, padding=1)
            if downsample else nn.Identity()
        )

    def forward(self, x, t_emb):
        skips = []
        for block in self.blocks:
            x = block(x, t_emb)
            skips.append(x)
        x = self.attn(x)
        x = self.downsample(x)
        return x, skips

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_blocks=2, upsample=True, use_attention=False):
        super().__init__()
        self.upsample = (
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
            if upsample else nn.Identity()
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(in_ch + out_ch, out_ch, time_emb_dim)
            for _ in range(num_blocks)
        ])
        self.attn = SelfAttention2D(out_ch) if use_attention else nn.Identity()

    def forward(self, x, skips, t_emb):
        x = self.upsample(x)
        for block in self.blocks:
            if skips:
                x = torch.cat([x, skips.pop()], dim=1)
            x = block(x, t_emb)
        x = self.attn(x)
        return x

class MidBlock(nn.Module):
    def __init__(self, channels, time_emb_dim, num_blocks=2):
        super().__init__()
        self.blocks = nn.ModuleList([
            ResidualBlock(channels, channels, time_emb_dim)
            for _ in range(num_blocks)
        ])
        self.attn = SelfAttention2D(channels)

    def forward(self, x, t_emb):
        for block in self.blocks:
            x = block(x, t_emb)
        x = self.attn(x)
        return x


## 10. 完整 UNet 与采样辅助函数

这部分对应原始脚本中的 `EnhancedUNet`、采样函数和图片保存函数。

In [ ]:
class EnhancedUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=128, time_emb_dim=512, num_res_blocks=2):
        super().__init__()

        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(base_ch),
            nn.Linear(base_ch, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        self.init_conv = nn.Conv2d(in_ch, base_ch, kernel_size=3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch, time_emb_dim, num_res_blocks, downsample=False)
        self.down2 = DownBlock(base_ch, base_ch * 2, time_emb_dim, num_res_blocks)
        self.down3 = DownBlock(base_ch * 2, base_ch * 4, time_emb_dim, num_res_blocks)
        self.down4 = DownBlock(base_ch * 4, base_ch * 8, time_emb_dim, num_res_blocks, use_attention=True)

        self.mid = MidBlock(base_ch * 8, time_emb_dim, num_res_blocks * 2)

        self.up4 = UpBlock(base_ch * 8, base_ch * 4, time_emb_dim, num_res_blocks, use_attention=True)
        self.up3 = UpBlock(base_ch * 4, base_ch * 2, time_emb_dim, num_res_blocks)
        self.up2 = UpBlock(base_ch * 2, base_ch, time_emb_dim, num_res_blocks)
        self.up1 = UpBlock(base_ch, base_ch, time_emb_dim, num_res_blocks, upsample=False)

        self.final = nn.Sequential(
            nn.GroupNorm(8, base_ch),
            nn.SiLU(),
            nn.Conv2d(base_ch, in_ch, kernel_size=3, padding=1),
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        x = self.init_conv(x)

        skips = []
        x, s1 = self.down1(x, t_emb); skips.extend(s1)
        x, s2 = self.down2(x, t_emb); skips.extend(s2)
        x, s3 = self.down3(x, t_emb); skips.extend(s3)
        x, s4 = self.down4(x, t_emb); skips.extend(s4)

        x = self.mid(x, t_emb)

        x = self.up4(x, skips, t_emb)
        x = self.up3(x, skips, t_emb)
        x = self.up2(x, skips, t_emb)
        x = self.up1(x, skips, t_emb)

        return self.final(x)

def save_samples(x, epoch):
    out = to_display_range(x)
    grid = utils.make_grid(out, nrow=int(math.sqrt(out.shape[0]) + 0.999), padding=2)
    filename = SAVE_DIR / f"sample_epoch_{epoch:03d}.png"
    utils.save_image(grid, filename)
    print(f"[saved] {filename}")

@torch.no_grad()
def sample_flow(model, n_samples=8, steps=100, device=DEVICE):
    model.eval()
    x = torch.randn(n_samples, 3, image_size, image_size, device=device)
    dt = 1.0 / steps
    for i in tqdm(range(steps), desc="sampling", leave=False):
        t = torch.full((n_samples,), float(i) / steps, device=device, dtype=torch.float32)
        u = model(x, t)
        x = x + u * dt
    model.train()
    return x.clamp(-1, 1)

def show_generated_samples(x, title="Generated Samples"):
    show_tensor_images(x, title=title)


## 11. 初始化模型、优化器与损失函数

为了更适合 Colab GPU，这里默认启用了 AMP 混合精度（如果当前设备是 CUDA）。

In [ ]:
model = EnhancedUNet(in_ch=3, base_ch=base_ch, time_emb_dim=512, num_res_blocks=2).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scaler = GradScaler(enabled=use_amp)
mse = nn.MSELoss()

num_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {num_params / 1e6:.2f}M")


## 12. 开始训练

训练逻辑仍然沿用原始脚本中的 Flow Matching 目标：
- 从标准高斯采样 `x_0`
- 从真实数据采样 `z`
- 线性构造中间态 `x_t = t z + (1 - t) x_0`
- 监督目标向量场 `u_target = z - x_0`


In [ ]:
print("Starting training... device:", DEVICE)

loss_history = []
global_step = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")
    for step, z in enumerate(pbar, start=1):
        z = z.to(DEVICE, non_blocking=True)
        B = z.shape[0]

        x_0 = torch.randn_like(z)
        t = torch.rand(B, device=DEVICE, dtype=torch.float32)
        t_broadcast = t.view(B, 1, 1, 1)

        # 构造中间时刻样本 x_t
        x_t = t_broadcast * z + (1.0 - t_broadcast) * x_0

        # Flow Matching 的监督目标向量场
        u_target = (z - x_0).detach()

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=use_amp):
            pred = model(x_t, t)
            loss = mse(pred, u_target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        avg_loss = running_loss / step
        pbar.set_postfix(loss=f"{avg_loss:.6f}")
        global_step += 1

    epoch_loss = running_loss / len(train_loader)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch + 1:03d} | avg_loss = {epoch_loss:.6f}")

    if (epoch + 1) % sample_every == 0:
        samples = sample_flow(model, n_samples=num_sample_images, steps=flow_steps, device=DEVICE)
        save_samples(samples, epoch + 1)
        show_generated_samples(samples, title=f"Generated @ Epoch {epoch + 1}")

    ckpt = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "global_step": global_step,
        "epoch": epoch + 1,
        "config": {
            "image_size": image_size,
            "base_ch": base_ch,
            "flow_steps": flow_steps,
        },
    }
    ckpt_path = SAVE_DIR / f"flowmatch_ckpt_epoch_{epoch + 1:03d}.pt"
    torch.save(ckpt, ckpt_path)
    print(f"[saved checkpoint] {ckpt_path}")

print("Training finished.")


## 13. 查看训练曲线与输出文件

这一格用来快速确认训练是否在下降，以及检查采样图和 checkpoint 是否已经生成。

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history, marker="o")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.show()

print("输出目录中的文件：")
for path in sorted(SAVE_DIR.iterdir()):
    print(" -", path.name)


## 14. 如何调回接近原始脚本的配置

如果你想更接近原始 `flowmatching.py`，可以把超参数改成下面这样：

- `image_size = 104`
- `batch_size = 32`
- `base_ch = 128`
- `num_epochs = 100`
- `flow_steps = 200`
- `max_train_images = None`

但这套配置对 Colab 免费 GPU 更吃内存和时间，建议先确认轻量版流程能完整跑通，再逐步放大。